In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path

RAW = Path("../Data/raw")
PROCESSED = Path("../Data/processed")
DASH = PROCESSED / "dashboard"
DASH.mkdir(exist_ok=True)

final = pd.read_csv(PROCESSED / "final_buurt_risk.csv")
print("Buurten in the final table:", len(final))

g = gpd.read_file(RAW / "External" / "cbs_buurten_2023_simplified.geojson")
g = g[g["buurtcode"].isin(final["buurtcode"])][["buurtcode", "geometry"]]
print("Buurten with a shape:", len(g))

# simplify in metres (Dutch grid), then back to longitude/latitude for the web map
g = g.to_crs("EPSG:28992")
g["geometry"] = g.simplify(25, preserve_topology=True)
g = g.to_crs("EPSG:4326")

out = DASH / "buurten_liander.geojson"
g.to_file(out, driver="GeoJSON")
print(f"Saved: {out.name}  ({out.stat().st_size / 1e6:.1f} MB)")

Buurten in the final table: 5620
Buurten with a shape: 5620
Saved: buurten_liander.geojson  (6.2 MB)


In [2]:
# ---------- ENTSO-E ----------
ENT = RAW / "1 EnstoE" / "Actual_Load Raw" / "entsoe_nl_2019_to_present.csv"
with open(ENT, encoding="utf-8-sig") as f:
    header = f.readline()
sep = ";" if header.count(";") > header.count(",") else ","
cols = ["Date", "actual_load", "solar_actual_aggregated",
        "wind_onshore_actual_aggregated", "wind_offshore_actual_aggregated"]
e = pd.read_csv(ENT, sep=sep, usecols=cols, decimal="," if sep == ";" else ".")
e["ts"] = pd.to_datetime(e["Date"], utc=True).dt.tz_convert("Europe/Amsterdam")
e = e.rename(columns={"actual_load": "load_mw", "solar_actual_aggregated": "solar_mw"})
e["wind_mw"] = e["wind_onshore_actual_aggregated"] + e["wind_offshore_actual_aggregated"]
e["year"], e["month"], e["hour"] = e["ts"].dt.year, e["ts"].dt.month, e["ts"].dt.hour
print("ENTSO-E from", e["ts"].min(), "to", e["ts"].max(), "|", len(e), "quarter-hours")

# 1. per year
yearly = e.groupby("year").agg(
    quarter_hours=("load_mw", "count"),
    energy_twh=("load_mw", lambda s: s.sum() * 0.25 / 1e6),   # MW x 0.25 h -> MWh -> TWh
    mean_mw=("load_mw", "mean"),
    peak_mw=("load_mw", "max"),
)
yearly["peak_time"] = e.loc[e.groupby("year")["load_mw"].idxmax(), "ts"].dt.strftime("%Y-%m-%d %H:%M").values
yearly["peak_to_average"] = yearly["peak_mw"] / yearly["mean_mw"]
yearly["complete_year"] = yearly["quarter_hours"] > 34000
print("\nPer year:")
print(yearly.round(2).to_string())

# 2. per month
monthly = e.groupby(["year", "month"]).agg(
    mean_mw=("load_mw", "mean"), peak_mw=("load_mw", "max"),
    solar_mean_mw=("solar_mw", "mean"), wind_mean_mw=("wind_mw", "mean"),
).reset_index()

# 3. average day per month (complete years 2019-2025)
profile = (e[e["year"].between(2019, 2025)]
           .groupby(["month", "hour"])[["load_mw", "solar_mw"]].mean().reset_index())

yearly.reset_index().to_csv(DASH / "entsoe_yearly.csv", index=False)
monthly.to_csv(DASH / "entsoe_monthly.csv", index=False)
profile.to_csv(DASH / "entsoe_profile_month_hour.csv", index=False)

# ---------- EV trend (national) ----------
cars = pd.read_excel(RAW / "2 CBS" / "3 Passenger car fuel.xlsx", header=None)
years = cars.iloc[2, 3:11].astype(int).tolist()
rows = cars.iloc[5:14]
rows = rows[rows[0].notna()]
ev = rows[[0] + list(range(3, 11))].set_index(0)
ev.columns = years
ev = ev.T.reset_index().rename(columns={"index": "year"})
ev.columns = [str(c).strip() for c in ev.columns]
ev.to_csv(DASH / "cars_fuel_national.csv", index=False)
print("\nEV table columns:", ev.columns.tolist())
print("\nSaved: entsoe_yearly.csv, entsoe_monthly.csv, entsoe_profile_month_hour.csv, cars_fuel_national.csv")

ENTSO-E from 2019-01-01 00:00:00+01:00 to 2026-09-16 14:15:00+02:00 | 270293 quarter-hours

Per year:
      quarter_hours  energy_twh   mean_mw   peak_mw         peak_time  peak_to_average  complete_year
year                                                                                                 
2019          35040      113.90  13002.41  18194.19  2019-01-22 17:30             1.40           True
2020          35136      108.15  12311.82  17843.75  2020-12-09 17:15             1.45           True
2021          35040      106.39  12144.72  17355.17  2021-01-04 17:30             1.43           True
2022          35040      100.39  11459.68  17786.31  2022-12-14 17:30             1.55           True
2023          35040      109.40  12488.02  18980.16  2023-12-04 17:15             1.52           True
2024          35136      115.04  13097.09  19577.23  2024-01-08 17:30             1.49           True
2025          35040      116.90  13345.13  19631.97  2025-11-19 17:30             

In [3]:
from pathlib import Path

files = [(f.name, f.stat().st_size / 1e6) for f in Path("../Data/processed").rglob("*") if f.is_file()]
for name, mb in sorted(files, key=lambda x: -x[1])[:15]:
    print(f"{mb:7.1f} MB   {name}")

   89.8 MB   liander_electricity_2019_2026.csv
   20.7 MB   model_a_table.csv
   18.2 MB   pc6_to_buurt_2023.csv
   17.5 MB   cbs_kwb_buurt_2019_2026.csv
    9.4 MB   transformer_projection_2026.csv
    7.7 MB   cbs_buurt_panel_2023codes.csv
    6.2 MB   buurten_liander.geojson
    5.9 MB   buurt_year_electricity.csv
    5.4 MB   transformers_buurt2023.csv
    3.7 MB   buurt_year_weather.csv
    2.6 MB   final_buurt_risk.csv
    2.4 MB   model_b_table.csv
    2.3 MB   final_buurt_risk_2027.csv
    1.2 MB   buurt_stress_predictions.csv
    0.4 MB   buurt_knmi_station.csv


In [1]:
import importlib.metadata as md

packages = ["pandas", "numpy", "openpyxl", "matplotlib", "scipy", "scikit-learn",
            "lightgbm", "shap", "geopandas", "pyogrio", "pyproj", "shapely",
            "streamlit", "plotly", "ipykernel"]

lines = [f"{p}=={md.version(p)}" for p in packages]
with open("../requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

print("\n".join(lines))

pandas==2.3.3
numpy==2.0.2
openpyxl==3.1.5
matplotlib==3.9.4
scipy==1.13.1
scikit-learn==1.6.1
lightgbm==4.6.0
shap==0.49.1
geopandas==1.0.1
pyogrio==0.11.1
pyproj==3.6.1
shapely==2.0.7
streamlit==1.50.0
plotly==7.1.0
ipykernel==6.31.0
